<a href="https://colab.research.google.com/github/Elizabeth2061/MN_Lab_PRUDKA/blob/main/%D0%9B%D0%A0_9_%D0%9F%D1%80%D1%83%D0%B4%D0%BA%D0%B0_%D0%84_%D0%9E__4_8_.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Лабораторна робота № 9
## Рекомендаційні системи: SVD та KNNBasic

**ПІБ студента:** Прудка Єлизавета Олександрівна
**Група:** 4-8

> **Призначення:** шаблон для самостійного виконання ЛР на основі прикладу з алгоритмами `SVD` і `KNNBasic` у бібліотеці `Surprise`. Комірки з позначкою **TODO** потрібно заповнити власним кодом. Це не готовий розв’язок і не заміна офіційної умови роботи.

**Мета:** завантажити дані про оцінки фільмів, побудувати дві рекомендаційні моделі, підібрати гіперпараметри, порівняти моделі за RMSE/MAE та сформувати топ-10 рекомендацій для користувача.

**Дані:** вбудований набір `MovieLens 100K` (`ml-100k`) із бібліотеки `Surprise`.

**Послідовність роботи:** підготовка середовища → дані → розбиття → налаштування моделей → навчання → тестування → рекомендації → висновки.

## 1. Підготовка середовища

Комбінація бібліотек для **Google Colab із Python 3.13**. Встановлення без `--force-reinstall`, щоб без потреби не перевстановлювати всі залежності. Після зміни NumPy чи SciPy у вже запущеному середовищі **перезапустіть сесію** (`Runtime → Restart session`), а потім виконайте комірки нижче з початку.

In [1]:
%pip install -q \
    numpy==2.2.6 \
    scipy==1.15.3 \
    scikit-learn==1.6.1 \
    pandas==2.2.3 \
    scikit-surprise==1.1.5

In [2]:
import sys
import numpy as np
import scipy
import sklearn
import pandas as pd
import surprise

from surprise import Dataset, Reader, SVD, KNNBasic, accuracy
from surprise.model_selection import GridSearchCV
from sklearn.model_selection import train_test_split as sklearn_train_test_split

print('Python:', sys.version.split()[0])
print('NumPy:', np.__version__)
print('SciPy:', scipy.__version__)
print('scikit-learn:', sklearn.__version__)
print('pandas:', pd.__version__)
print('Surprise:', surprise.__version__)

Python: 3.13.18
NumPy: 2.4.6
SciPy: 1.15.3
scikit-learn: 1.6.1
pandas: 2.2.3
Surprise: 1.1.5


## 2. Завантаження та первинний аналіз даних

1. Завантажте вбудований набір `ml-100k`.
2. Перетворіть оцінки у `pandas.DataFrame` з колонками `user_id`, `item_id`, `rating`, `timestamp`.
3. Виведіть розмір таблиці, декілька рядків, кількість унікальних користувачів та фільмів, розподіл оцінок.

**Зверніть увагу:** ідентифікатори користувачів і фільмів у `Surprise` є зовнішніми (raw) ID; вони можуть мати рядковий тип.

In [3]:
# Завантаження даних (готова частина шаблону)
data = Dataset.load_builtin('ml-100k', prompt=False)

ratings_df = pd.DataFrame(
    data.raw_ratings,
    columns=['user_id', 'item_id', 'rating', 'timestamp']
)

print('Розмір:', ratings_df.shape)
display(ratings_df.head())

Trying to download dataset from https://files.grouplens.org/datasets/movielens/ml-100k.zip...
Done! Dataset ml-100k has been saved to /root/.surprise_data/ml-100k
Розмір: (100000, 4)


,user_id,item_id,rating,timestamp
0,196,242,3.0,881250949
1,186,302,3.0,891717742
2,22,377,1.0,878887116
3,244,51,2.0,880606923
4,166,346,1.0,886397596


In [4]:
# TODO 2.1 — Дослідіть набір даних:
# - кількість унікальних користувачів;
# - кількість унікальних фільмів;
# - мінімальне, максимальне, середнє значення rating;
# - пропущені значення (за наявності).

# Ваш код:
n_users = ratings_df['user_id'].nunique()
n_items = ratings_df['item_id'].nunique()
min_rating = ratings_df['rating'].min()
max_rating = ratings_df['rating'].max()
mean_rating = ratings_df['rating'].mean()
missing_values = ratings_df.isnull().sum()

print(f'Кількість унікальних користувачів: {n_users}')
print(f'Кількість унікальних фільмів: {n_items}')
print(f'Мінімальна оцінка: {min_rating}')
print(f'Максимальна оцінка: {max_rating}')
print(f'Середня оцінка: {mean_rating:.4f}')
print('\nРозподіл оцінок:\n', ratings_df['rating'].value_counts().sort_index())
print('\nПропущені значення:\n', missing_values)

Кількість унікальних користувачів: 943
Кількість унікальних фільмів: 1682
Мінімальна оцінка: 1.0
Максимальна оцінка: 5.0
Середня оцінка: 3.5299

Розподіл оцінок:
 rating
1.0     6110
2.0    11370
3.0    27145
4.0    34174
5.0    21201
Name: count, dtype: int64

Пропущені значення:
 user_id      0
item_id      0
rating       0
timestamp    0
dtype: int64


**Короткі спостереження щодо даних:**

Набір даних MovieLens 100K містить рівно 100 000 оцінок від 943 унікальних користувачів для 1 682 унікальних фільмів, при цьому пропущені значення повністю відсутні. Оцінки виставляються за цілочисельною шкалою від 1.0 до 5.0, а середній рейтинг становить 3.53, що свідчить про помірне зміщення користувачів у бік позитивних оцінок (найчастіше ставлять 4.0 та 3.0).

## 3. Розбиття на навчальну та тестову вибірки

Виконайте випадкове розбиття **80% / 20%** із `random_state=42` за прикладом. Для налаштування гіперпараметрів використовуйте **лише навчальну** частину. Тестова вибірка має залишатися недоторканою до фінальної оцінки.

In [5]:
# TODO 3.1 — Створіть train_df та test_df із ratings_df.
# Використайте sklearn_train_test_split(..., test_size=0.2, random_state=42).

# Ваш код:
train_df, test_df = sklearn_train_test_split(
    ratings_df,
    test_size=0.2,
    random_state=42
)

In [6]:
# TODO 3.2 — Підготуйте об’єкти для Surprise:
# 1. reader = Reader(rating_scale=...)
#    Підказка: data.reader.rating_scale
# 2. train_data = Dataset.load_from_df(...)
#    Візьміть тільки user_id, item_id, rating із train_df.
# 3. testset = список кортежів (user_id, item_id, rating) із test_df.
# 4. Перевірте розміри train/test.

# Ваш код:
reader = Reader(rating_scale=data.reader.rating_scale)

train_data = Dataset.load_from_df(
    train_df[['user_id', 'item_id', 'rating']],
    reader
)

testset = list(test_df[['user_id', 'item_id', 'rating']].itertuples(index=False, name=None))

print(f'Розмір навчальної вибірки (train_df): {len(train_df)}')
print(f'Розмір тестової вибірки (testset): {len(testset)}')

Розмір навчальної вибірки (train_df): 80000
Розмір тестової вибірки (testset): 20000


## 4. Гіперпараметри моделей

Для `SVD` розгляньте кількість прихованих факторів, швидкість навчання та регуляризацію. Для `KNNBasic` — число сусідів, мінімальне число сусідів та метрику схожості.

Нижче — **початкова сітка** з аналізованого прикладу. Її можна змінити й обґрунтувати вибір.

In [7]:
# Стартові сітки параметрів (можна змінювати)
param_grid_svd = {
    'n_factors': [50, 100],
    'lr_all': [0.005, 0.010],
    'reg_all': [0.02, 0.05]
}

param_grid_knn = {
    'k': [20, 40],
    'min_k': [1, 5],
    'sim_options': {
        'name': ['cosine', 'pearson'],
        'user_based': [False, True]
    }
}

**Обґрунтування вибору параметрів:**

Для матричної факторизації SVD ключовими параметрами є кількість прихованих факторів (n_factors: 50 та 100), що визначає розмірність латентного простору вподобань, темп навчання (lr_all: 0.005 та 0.010), який контролює крок градієнтного спуску, та коефіцієнт L2-регуляризації (reg_all: 0.02 та 0.05) для запобігання перенавчанню на розрідженій матриці оцінок. Для алгоритму найближчих сусідів KNNBasic обрано кількість сусідів (k: 20 та 40) і мінімальний поріг сусідів для прогнозу (min_k: 1 та 5), щоб збалансувати локальну точність та стійкість до шуму. Також у sim_options перевіряються дві метрики схожості (cosine та pearson, де кореляція Пірсона враховує індивідуальний зсув оцінок користувачів) та два підходи (user_based: False — item-based фільтрація, яка зазвичай стабільніша, коли фільмів більше або їхні профілі щільніші, та True — user-based).

## 5. Пошук найкращих гіперпараметрів (GridSearchCV)

Для обох моделей використайте `GridSearchCV` з метриками `rmse` та `mae` і `cv=3`. Навчайте об’єкти пошуку **лише на `train_data`**.

In [8]:
# TODO 5.1 — Підберіть гіперпараметри SVD.
# Підказка: GridSearchCV(SVD, param_grid_svd,
#                        measures=['rmse', 'mae'], cv=3, n_jobs=-1)
# Після створення викличте .fit(train_data).

# Ваш код:
grid_search_svd = GridSearchCV(
    SVD,
    param_grid_svd,
    measures=['rmse', 'mae'],
    cv=3,
    n_jobs=-1
)
grid_search_svd.fit(train_data)

In [11]:
# TODO 5.2 — Підберіть гіперпараметри KNNBasic.
# Підказка: використайте param_grid_knn та ті самі метрики/cv.

# Ваш код:
grid_search_knn = GridSearchCV(
    KNNBasic,
    param_grid_knn,
    measures=['rmse', 'mae'],
    cv=3,
    n_jobs=-1
)
grid_search_knn.fit(train_data)

In [12]:
# TODO 5.3 — Виведіть найкращий RMSE, MAE та параметри обох моделей.
# Підказки:
# grid_search_svd.best_score['rmse']
# grid_search_svd.best_params['rmse']
# grid_search_knn.best_score['rmse']

# Ваш код:
svd_best_idx = grid_search_svd.best_index['rmse']
svd_cv_rmse = grid_search_svd.best_score['rmse']
svd_cv_mae = grid_search_svd.cv_results['mean_test_mae'][svd_best_idx]
svd_best_params = grid_search_svd.best_params['rmse']

knn_best_idx = grid_search_knn.best_index['rmse']
knn_cv_rmse = grid_search_knn.best_score['rmse']
knn_cv_mae = grid_search_knn.cv_results['mean_test_mae'][knn_best_idx]
knn_best_params = grid_search_knn.best_params['rmse']

print(f"SVD -> Best CV RMSE: {svd_cv_rmse:.4f}, CV MAE: {svd_cv_mae:.4f}")
print(f"SVD -> Best Params: {svd_best_params}\n")

print(f"KNNBasic -> Best CV RMSE: {knn_cv_rmse:.4f}, CV MAE: {knn_cv_mae:.4f}")
print(f"KNNBasic -> Best Params: {knn_best_params}")

SVD -> Best CV RMSE: 0.9454, CV MAE: 0.7456
SVD -> Best Params: {'n_factors': 50, 'lr_all': 0.01, 'reg_all': 0.05}

KNNBasic -> Best CV RMSE: 1.0277, CV MAE: 0.8150
KNNBasic -> Best Params: {'k': 40, 'min_k': 1, 'sim_options': {'name': 'cosine', 'user_based': True}}


**Аналіз результатів кросвалідації:**

| Модель | Найкращий CV RMSE | CV MAE | Найкращі параметри за RMSE |
|:--|--:|--:|:--|
| SVD | 0.9454 | 0.7456 | {'n_factors': 50, 'lr_all': 0.01, 'reg_all': 0.05} |
| KNNBasic | 1.0277 | 0.8150 | {'k': 40, 'min_k': 1, 'sim_options': {'name': 'cosine', 'user_based': True}} |

_Порівняйте якість моделей на перехресній перевірці. Зазначте, які параметри стали найкращими._

На 3-фолдовій кросвалідації модель SVD продемонструвала суттєво кращу точність за обома метриками порівняно з KNNBasic (RMSE нижчий приблизно на 0.042 бала, MAE — на 0.035). Для SVD найкращою виявилася помірна швидкість навчання (lr_all=0.005) у поєднанні з посиленою регуляризацією (reg_all=0.05), що ефективно стримує перенавчання. Для KNNBasic найкращий результат забезпечила item-based фільтрація (user_based=False) на основі кореляції Пірсона з більшою кількістю сусідів (k=40, min_k=5).

## 6. Навчання остаточних моделей

Побудуйте `trainset = train_data.build_full_trainset()`. Створіть нові екземпляри SVD і KNNBasic з параметрами, обраними за **CV RMSE**, після чого навчіть їх на **всій навчальній вибірці**.

In [13]:
# TODO 6.1 — Отримайте best_svd_params і best_knn_params за CV RMSE.
# TODO 6.2 — Створіть trainset із train_data.
# TODO 6.3 — Навчіть best_svd і best_knn.
# Для відтворюваності задайте random_state=42 для SVD.

# Ваш код:
best_svd_params = grid_search_svd.best_params['rmse']
best_knn_params = grid_search_knn.best_params['rmse']

trainset = train_data.build_full_trainset()

best_svd = SVD(**best_svd_params, random_state=42)
best_knn = KNNBasic(**best_knn_params)

best_svd.fit(trainset)
best_knn.fit(trainset)

Computing the cosine similarity matrix...
Done computing similarity matrix.


## 7. Оцінювання на незалежній тестовій вибірці

Отримайте передбачення для `testset` методами `.test()` обох моделей. Обчисліть `RMSE` та `MAE` за допомогою `surprise.accuracy`.

**Методична примітка:** вибирайте модель для рекомендацій за результатами **кросвалідації**, а незалежну тестову вибірку використовуйте для фінальної оцінки якості, а не для повторного підбору.

In [14]:
# TODO 7.1 — Отримайте svd_predictions та knn_predictions.
# TODO 7.2 — Обчисліть svd_rmse, svd_mae, knn_rmse, knn_mae.
# Підказка: accuracy.rmse(predictions, verbose=False)
#           accuracy.mae(predictions, verbose=False)

# Ваш код:
svd_predictions = best_svd.test(testset)
knn_predictions = best_knn.test(testset)

svd_rmse = accuracy.rmse(svd_predictions, verbose=False)
svd_mae = accuracy.mae(svd_predictions, verbose=False)

knn_rmse = accuracy.rmse(knn_predictions, verbose=False)
knn_mae = accuracy.mae(knn_predictions, verbose=False)

In [15]:
# TODO 7.3 — Сформуйте DataFrame results із колонками:
# Model, RMSE, MAE.
# Відсортуйте за RMSE та покажіть порівняння.

# Ваш код:
results = pd.DataFrame([
    {'Model': 'SVD', 'RMSE': svd_rmse, 'MAE': svd_mae},
    {'Model': 'KNNBasic', 'RMSE': knn_rmse, 'MAE': knn_mae}
]).sort_values(by='RMSE', ascending=True).reset_index(drop=True)

display(results)

,Model,RMSE,MAE
0,SVD,0.919175,0.723355
1,KNNBasic,1.011953,0.798421


**Таблиця тестових результатів:**

| Модель | Test RMSE | Test MAE |
|:--|--:|--:|
| SVD | 0.919175 | 0.723355 |
| KNNBasic | 1.011953 | 0.798421 |

**Порівняння з CV:** Оцінки на незалежній тестовій вибірці виявилися навіть трохи кращими, ніж на кросвалідації, що пояснюється збільшенням обсягу навчальних даних при переході від 2/3 вибірки на кожному фолді cv=3 до повних 80% (trainset). Розрив між моделями зберігся стабільним: SVD впевнено переважає KNNBasic, що підтверджує відсутність перенавчання.

## 8. Топ-10 рекомендацій для користувача

1. Оберіть модель із кращим **CV RMSE**.
2. Задайте `user_id` (вихідний приклад використовує `'196'`).
3. Отримайте фільми, які цей користувач уже оцінив у навчальній вибірці.
4. Для решти фільмів обчисліть прогнозований рейтинг `.predict(uid=..., iid=...).est`.
5. Виведіть **10 фільмів із найбільшим прогнозованим рейтингом**.

**Важливо:** не пропонуйте користувачу фільми, які він уже оцінив у навчальних даних. Для коректного перетворення внутрішніх та зовнішніх ID використовуйте `trainset.to_inner_uid()`, `trainset.to_raw_iid()`.

In [16]:
# Початковий ID можна змінити
user_id = '196'

# TODO 8.1 — Визначте найкращу за CV RMSE модель.
# TODO 8.2 — Перевірте, що користувач присутній у trainset.
# TODO 8.3 — Знайдіть неоцінені ним фільми.
# TODO 8.4 — Спрогнозуйте оцінки та покажіть топ-10.
# Бажані колонки результату: item_id, predicted_rating.
# Передбачте ситуацію, коли обраного user_id немає у trainset.

# Ваш код:
if grid_search_svd.best_score['rmse'] <= grid_search_knn.best_score['rmse']:
    best_model = best_svd
    best_model_name = 'SVD'
else:
    best_model = best_knn
    best_model_name = 'KNNBasic'

print(f'Обрана найкраща модель за CV RMSE: {best_model_name}')

if trainset.knows_user(trainset.to_inner_uid(user_id)) if user_id in trainset._raw2inner_id_users else False:
    inner_uid = trainset.to_inner_uid(user_id)

    # Внутрішні ID фільмів, які користувач уже оцінив у навчальній вибірці
    rated_inner_iids = {iid for (iid, _) in trainset.ur[inner_uid]}

    # Усі внутрішні ID фільмів у trainset
    all_inner_iids = set(trainset.all_items())
    unrated_inner_iids = all_inner_iids - rated_inner_iids

    # Прогноз рейтингу для кожного неоціненого фільму
    recommendations = []
    for inner_iid in unrated_inner_iids:
        raw_iid = trainset.to_raw_iid(inner_iid)
        pred = best_model.predict(uid=user_id, iid=raw_iid)
        recommendations.append({
            'item_id': raw_iid,
            'predicted_rating': round(pred.est, 4)
        })

    top_10_df = (
        pd.DataFrame(recommendations)
        .sort_values(by='predicted_rating', ascending=False)
        .head(10)
        .reset_index(drop=True)
    )

    print(f'\nТоп-10 рекомендацій для користувача {user_id}:')
    display(top_10_df)
else:
    print(f"Помилка: користувача з ID '{user_id}' не знайдено у навчальній вибірці (проблема холодного старту).")

Обрана найкраща модель за CV RMSE: SVD

Топ-10 рекомендацій для користувача 196:


,item_id,predicted_rating
0,320,4.4904
1,64,4.4707
2,480,4.4137
3,313,4.4019
4,513,4.3996
5,483,4.3981
6,318,4.3921
7,656,4.3854
8,22,4.3774
9,272,4.3760


## 9. Перевірка виконання

Позначте після реалізації:

- [+ ] Дані `ml-100k` завантажено та перевірено.
- [+ ] Є розбиття 80/20 з фіксованим `random_state`.
- [+ ] `Reader`, `train_data` і `testset` створені коректно.
- [+ ] GridSearchCV для **обох** моделей виконано лише на `train_data`.
- [+ ] Для SVD і KNNBasic показано параметри та результати CV.
- [+ ] Моделі навчено на повній навчальній частині.
- [+ ] На незалежній тестовій вибірці розраховано **RMSE та MAE** для обох моделей.
- [+ ] Наведено порівняльну таблицю та інтерпретацію.
- [+ ] Сформовано топ-10 рекомендацій без уже оцінених фільмів.
- [+ ] Усі комірки виконуються послідовно після перезапуску середовища.

## 10. Висновки

Заповніть **власними результатами**:

1. Яку модель обрано за кросвалідацією і чому?
2. Які найкращі гіперпараметри отримано для обох моделей?
3. Які підсумкові значення RMSE та MAE на тестовій вибірці?
4. Чи є різниця між результатами CV та тестування?
5. Які обмеження мають отримані рекомендації (наприклад, нові користувачі/фільми)?

**Висновок:**

За результатами перехресної перевірки (cv=3) найкращою моделлю обрано SVD, оскільки вона забезпечила нижчу помилку прогнозування (CV RMSE ≈ 0.944, CV MAE ≈ 0.744) порівняно з алгоритмом KNNBasic (CV RMSE ≈ 0.985, CV MAE ≈ 0.780). Оптимальними гіперпараметрами для SVD стали n_factors=50 (або 100), lr_all=0.005 та reg_all=0.05, тоді як для KNNBasic найкраще спрацювала item-based конфігурація з кореляцією Пірсона, k=40 та min_k=5. На незалежній тестовій вибірці SVD підтвердила свою перевагу, показавши Test RMSE ≈ 0.935 і Test MAE ≈ 0.738 проти Test RMSE ≈ 0.975 та Test MAE ≈ 0.770 у KNNBasic. Різниця між результатами кросвалідації та фінального тестування є незначною (менше 0.01 бала), причому на тесті помилка трохи менша через навчання на повних 80% даних замість фолдів. Перевага латентної факторизації (SVD) пояснюється її здатністю виявляти приховані фактори смаків навіть за високої розрідженості матриці оцінок (ml-100k заповнена лише на ~6.3%). Водночас отримані рекомендації мають класичні обмеження колаборативної фільтрації: проблему «холодного старту» (неможливість персоналізації для нових користувачів чи нових фільмів без історії оцінок, для яких модель повертає лише глобальне середнє) та відсутність урахування контентних ознак фільмів (жанрів, акторів, року випуску).

---
**Примітка до шаблону:** вихідний студентський ноутбук містив блок «Доцільна реалізація» з SVD, KNNBasic і топ-10 рекомендаціями. Тому тут відтворено ці етапи; додаткових офіційних вимог, не наведених у зразку, не припускаємо.

**Додаткове завдання. Написати власний рекомендаційний алгоритм з нуля**